In [ ]:
import pandas as pd
import sqlite3
import os

def run_ecommerce_etl_pipeline():
    print("🚀 Initiating Enterprise E-Commerce ETL Pipeline...")

    # 1. Locate and Ingest the dataset
    csv_files = [f for f in os.listdir('.') if f.endswith('.csv')]
    if not csv_files:
        raise FileNotFoundError("❌ Please upload your 1M row Amazon CSV dataset into the Colab side panel.")

    target_file = csv_files[0]
    print(f"📦 Loading Raw Source Ledger: '{target_file}'...")
    df = pd.read_csv(target_file)
    print(f"✅ Ingestion complete. Raw Volume: {len(df):,} records.")

    # Clean up column names just in case there are trailing spaces
    df.columns = df.columns.str.strip()

    # 2. Establish High-Performance In-Memory SQLite Database Warehouse
    print("🗄️ Initializing OLAP Database Warehouse Engine...")
    conn = sqlite3.connect("amazon_data_warehouse.db")
    cursor = conn.cursor()

    # 3. TRANSFORM: Deconstruct and Normalize Flat File into Star Schema Tables
    print("🧬 Normalizing dataset into an optimized Star Schema architecture...")

    # Dimension 1: Users
    dim_users = df[['user_id', 'location', 'device']].drop_duplicates(subset=['user_id'])

    # Dimension 2: Products
    dim_products = df[['product_id', 'category', 'subcategory', 'brand', 'price']].drop_duplicates(subset=['product_id'])

    # Dimension 3: Sellers
    dim_sellers = df[['seller_id', 'seller_rating']].drop_duplicates(subset=['seller_id'])

    # Fact Table: Sales (Contains foreign keys and transactional metrics)
    fact_sales = df[[
        'user_id', 'product_id', 'seller_id', 'purchase_date', 'discount',
        'final_price', 'rating', 'review_count', 'stock', 'shipping_time_days',
        'payment_method', 'is_returned', 'delivery_status'
    ]]

    # 4. LOAD: Write Clean Tables into the Warehouse
    print("📥 Loading normalized matrices into database warehouse tables...")
    dim_users.to_sql("dim_users", conn, if_exists="replace", index=False)
    dim_products.to_sql("dim_products", conn, if_exists="replace", index=False)
    dim_sellers.to_sql("dim_sellers", conn, if_exists="replace", index=False)
    fact_sales.to_sql("fact_sales", conn, if_exists="replace", index=False)

    print("🎯 Warehouse structure successfully instantiated and indexed.")
    print("-" * 60)

    # 5. EXECUTE EXHAUSTIVE SQL AUDITS (Analytical Core)
    print("📊 RUNNING ENTERPRISE METRIC QUERIES (PURE SQL)...")
    print("-" * 60)

    # QUERY 1: High-Value Logistic Performance & Operational Risk Audit
    # Tracks delivery speeds and return rates across major categories
    query_logistics = """
        SELECT
            p.category AS Category,
            COUNT(f.product_id) AS Total_Units_Sold,
            ROUND(AVG(f.shipping_time_days), 2) AS Avg_Shipping_Days,
            ROUND(AVG(f.is_returned) * 100, 2) AS Return_Rate_Percentage,
            ROUND(SUM(f.final_price), 2) AS Total_Gross_Revenue
        FROM fact_sales f
        JOIN dim_products p ON f.product_id = p.product_id
        GROUP BY p.category
        ORDER BY Total_Gross_Revenue DESC;
    """

    # QUERY 2: Merchant Performance Tiering & Revenue Contributions
    # Uses Window Functions to rank top brands within categories by revenue
    query_brands = """
        WITH BrandRevenue AS (
            SELECT
                p.category AS Category,
                p.brand AS Brand,
                SUM(f.final_price) AS Gross_Revenue,
                AVG(f.rating) AS Avg_Customer_Rating,
                ROW_NUMBER() OVER(PARTITION BY p.category ORDER BY SUM(f.final_price) DESC) as Rank
            FROM fact_sales f
            JOIN dim_products p ON f.product_id = p.product_id
            GROUP BY p.category, p.brand
        )
        SELECT Category, Brand, ROUND(Gross_Revenue, 2) as Revenue, ROUND(Avg_Customer_Rating, 2) as Rating
        FROM BrandRevenue
        WHERE Rank <= 2;
    """

    # Run and display Query 1
    print("\n📋 REPORT 1: LOGISTICS METRICS & REVENUE PROFILE BY CATEGORY")
    df_logistics = pd.read_sql_query(query_logistics, conn)
    print(df_logistics.to_string(index=False))

    # Run and display Query 2
    print("\n📋 REPORT 2: TOP 2 MARKETPLACE BRANDS PER CATEGORY (SQL WINDOW FUNCTIONS)")
    df_brands = pd.read_sql_query(query_brands, conn)
    print(df_brands.to_string(index=False))

    # Close connection cleanly
    conn.close()
    print("\n🏁 Pipeline executed flawlessly. Warehouse local file compiled.")

if __name__ == "__main__":
    run_ecommerce_etl_pipeline()

🚀 Initiating Enterprise E-Commerce ETL Pipeline...
📦 Loading Raw Source Ledger: 'amazon_ecommerce_1M.csv'...
✅ Ingestion complete. Raw Volume: 1,000,000 records.
🗄️ Initializing OLAP Database Warehouse Engine...
🧬 Normalizing dataset into an optimized Star Schema architecture...
📥 Loading normalized matrices into database warehouse tables...
🎯 Warehouse structure successfully instantiated and indexed.
------------------------------------------------------------
📊 RUNNING ENTERPRISE METRIC QUERIES (PURE SQL)...
------------------------------------------------------------

📋 REPORT 1: LOGISTICS METRICS & REVENUE PROFILE BY CATEGORY
   Category  Total_Units_Sold  Avg_Shipping_Days  Return_Rate_Percentage  Total_Gross_Revenue
Electronics            201475               3.17                   11.62         2.404890e+09
       Home            202528               3.16                   11.56         1.974469e+09
     Sports            198487               3.17                   11.64        